# services/source_select

> Service layer for file operations and audio extraction from video via FFmpeg plugin

In [1]:
#| default_exp services.source_select

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import asyncio
from typing import Dict, Any, Optional, List

from cjm_plugin_system.core.manager import PluginManager

## SourceSelectService

Thin wrapper around the FFmpeg plugin for file info and audio extraction. Follows the same pattern as `SegmentationService` and `AlignmentService`: constructor takes `PluginManager` + plugin name, execution goes through `manager.execute_plugin_async()`.

In [4]:
#| export
class SourceSelectService:
    """Service for file metadata and audio extraction from video."""

    def __init__(self,
                 plugin_manager: PluginManager,  # Plugin manager instance
                 plugin_name: str = "cjm-media-plugin-ffmpeg",  # FFmpeg plugin name
                ):
        self._manager = plugin_manager
        self._plugin_name = plugin_name

    def is_available(self) -> bool:  # True if plugin is loaded
        """Check if the FFmpeg plugin is loaded."""
        return self._manager.get_plugin(self._plugin_name) is not None

    def ensure_loaded(self,
                      config: Optional[Dict[str, Any]] = None,  # Plugin config override
                     ) -> bool:  # True if plugin is now loaded
        """Ensure the FFmpeg plugin is loaded, loading it if necessary."""
        if self.is_available():
            return True
        meta = self._manager.get_discovered_meta(self._plugin_name)
        if meta:
            return self._manager.load_plugin(meta, config or {})
        return False

    async def get_file_info(
        self,
        file_path: str,  # Path to media file
    ) -> Dict[str, Any]:  # MediaMetadata dict (duration, streams, etc.)
        """Get media file metadata via FFmpeg plugin."""
        if not self.is_available():
            raise RuntimeError(f"Plugin {self._plugin_name} not loaded")
        return await self._manager.execute_plugin_async(
            self._plugin_name,
            action="get_info",
            file_path=file_path,
        )

    async def extract_audio(
        self,
        video_path: str,  # Path to video file
    ) -> Dict[str, Any]:  # {job_id, output_path, duration, codec, stream_copy}
        """Extract audio from a video file via FFmpeg plugin (stream copy)."""
        if not self.is_available():
            raise RuntimeError(f"Plugin {self._plugin_name} not loaded")
        return await self._manager.execute_plugin_async(
            self._plugin_name,
            action="extract_audio",
            input_path=video_path,
        )

    def extract_audio_sync(
        self,
        video_path: str,  # Path to video file
    ) -> Dict[str, Any]:  # {job_id, output_path, duration, codec, stream_copy}
        """Synchronous wrapper for extract_audio."""
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(self.extract_audio(video_path))

In [5]:
#| hide
import nbdev; nbdev.nbdev_export()